## Python Replay Notebook

This notebook replays historical candles one bar at a time using the same VWAP probability band engine used by the live workflow.

It is not connected to TradingView. TradingView replay will require a separate Pine Script overlay later.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("✅ Project root added to Python path:")
print(PROJECT_ROOT)

In [ ]:
from src.config import CONFIG
from src.loaders import load_tradingview_csv, assign_sessions
from src.engine import EngineState, update_engine_state
from src.replay import run_replay

### Load historical data

This loads the configured historical candle file and assigns session ids before replay.

In [ ]:
csv_path = CONFIG["data_dir"] / CONFIG["csv_filename"]

if not csv_path.is_absolute():
    csv_path = PROJECT_ROOT / csv_path

print("CSV path:", csv_path)
print("Exists:", csv_path.exists())

df = load_tradingview_csv(csv_path)
df = assign_sessions(df, CONFIG["sessions"])

print("Rows:", len(df))
display(df.head())
display(df.tail())

### Load probability tables

The replay engine needs a calibrated probability table. The marginal table is used as a fallback when exact trend/context bins are sparse.

In [ ]:
tables_dir = PROJECT_ROOT / "artifacts" / "tables"

prob_table_path = tables_dir / "prob_table_trend.parquet"
marginal_table_path = tables_dir / "prob_table_marginal.parquet"

print("Trend table exists:", prob_table_path.exists(), prob_table_path)
print("Marginal table exists:", marginal_table_path.exists(), marginal_table_path)

prob_table = pd.read_parquet(prob_table_path)

if marginal_table_path.exists():
    marginal_table = pd.read_parquet(marginal_table_path)
else:
    marginal_table = prob_table

print("prob_table:", prob_table.shape)
print("marginal_table:", marginal_table.shape)

display(prob_table.head())